# AtlasRank M5 supervised training
Run the unchanged MiniLM configuration: ten-update sanity check, then continue the same optimizer/scheduler state to 574 updates only if checks pass. CPU-prestaged inputs; 3600 seconds new GPU allocation, 2700 active seconds, 300 finalization reserve. No development/test supervision.


In [ ]:
from pathlib import Path

ALLOCATION_STARTED_UTC = None  # Actual UTC epoch from L4 allocation, before upload/setup.
ARCHIVE_UPLOAD = Path(
    "/content/drive/MyDrive/AtlasRank/m5/staging-training-v1/m5-training-input-v8.zip"
)
APPROVED_MANIFEST = Path(
    "/content/drive/MyDrive/AtlasRank/m5/staging-training-v1/m5-training.approved.json"
)
WORK = Path("/content/atlasrank-m5-work")
INPUTS = WORK / "input"
VENV = WORK / "venv"
RECOVER_INTERRUPTED = False  # True only after confirming the prior Colab VM stopped.
if ALLOCATION_STARTED_UTC is None:
    raise PermissionError(
        "Record actual L4 allocation time and obtain one execution approval first"
    )

from google.colab import drive  # noqa: E402

drive.mount("/content/drive")

## Runtime discovery

This checks the native Colab Torch/CUDA build before installing anything. It rejects a non-L4, changed Torch build, insufficient RAM, or low free VRAM. Do not replace Colab's CUDA-enabled Torch.



In [ ]:
import os

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json
import sys

import psutil
import torch

torch.cuda.init()
free_vram, total_vram = torch.cuda.mem_get_info(0)
native_torch_path = str(torch.__file__)
discovery = {
    "python": sys.version,
    "torch": str(torch.__version__),
    "torch_cuda": torch.version.cuda,
    "torch_path": native_torch_path,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0),
    "free_vram_bytes": free_vram,
    "total_vram_bytes": total_vram,
    "available_ram_bytes": psutil.virtual_memory().available,
}
print(json.dumps(discovery, indent=2))
if (
    sys.version.split()[0] != "3.13.16"
    or str(torch.__version__) != "2.11.0+cu130"
    or torch.version.cuda != "13.0"
    or discovery["gpu"] != "NVIDIA L4"
    or not discovery["cuda_available"]
    or discovery["available_ram_bytes"] < 16 * 1024**3
    or free_vram < 8 * 1024**3
):
    raise RuntimeError("Runtime differs from approved M5 pilot; stop and report discovery")

In [ ]:
import hashlib
import json
import zipfile

approved = json.loads(APPROVED_MANIFEST.read_text())
if approved.get("execution_approved") is not True:
    raise PermissionError("The proposed manifest cannot execute training")
if (
    approved.get("input_archive_sha256")
    != "82e7b062eb50e229a541aa57edd131f1f9318b1661478fe00cfa8c36cb1ec596"
):
    raise ValueError("Approved manifest does not pin the reviewed local training export")
if ARCHIVE_UPLOAD.stat().st_size != approved.get("input_archive_bytes"):
    raise ValueError("Training export size differs from the approved manifest")
digest = hashlib.sha256()
with ARCHIVE_UPLOAD.open("rb") as stream:
    for block in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(block)
if digest.hexdigest() != approved["input_archive_sha256"]:
    raise ValueError("Training ZIP checksum differs from the approved pin")
if (
    approved["execution_environment"].get("gpu_name") != "NVIDIA L4"
    or approved["sampling_and_loss"].get("batch_size_query_groups") != 16
):
    raise ValueError("Manifest device/batch configuration differs")
with zipfile.ZipFile(ARCHIVE_UPLOAD) as archive:
    bootstrap = archive.read("source/src/atlasrank/retrieval/colab_transport.py")
    if archive.testzip() is not None:
        raise ValueError("Training input ZIP failed CRC validation")
namespace = {"__name__": "atlasrank_m5_transport_bootstrap"}
exec(compile(bootstrap, "<reviewed-colab-transport>", "exec"), namespace)
WORK.mkdir(parents=True, exist_ok=True)
namespace["safe_extract"](ARCHIVE_UPLOAD, INPUTS, approved["resource_limits"]["upload_bytes"])
export = json.loads((INPUTS / "EXPORT.json").read_text())
for relative, expected in export["source_sha256"].items():
    base = INPUTS if relative.startswith("environments/") else INPUTS / "source"
    candidate = base / relative
    actual = hashlib.sha256(candidate.read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f"Reviewed source/environment hash mismatch: {relative}")
sys.path.insert(0, str(INPUTS / "source/src"))
print("Verified training export:", approved["input_archive_sha256"])
print(
    "Train groups:",
    export["training_groups"],
    "selected products:",
    export["selected_product_identity_count"],
)

from atlasrank.training.configuration import parse_manifest, validate_execution_scope  # noqa: E402

scope = validate_execution_scope(parse_manifest(approved))
import threading  # noqa: E402
import time  # noqa: E402

from google.colab import runtime  # noqa: E402

DIAGNOSTIC_DEADLINE = ALLOCATION_STARTED_UTC + scope["max_new_allocation_seconds"]
if DIAGNOSTIC_DEADLINE - time.time() < 600:
    raise TimeoutError("Insufficient setup/active/finalization margin; stop and release VM")
if "_M5_RELEASE_TIMER" not in globals():
    _M5_RELEASE_TIMER = threading.Timer(max(0, DIAGNOSTIC_DEADLINE - time.time()), runtime.unassign)
    _M5_RELEASE_TIMER.daemon = True
    _M5_RELEASE_TIMER.start()
print("Verified execution scope:", scope)

## Pinned overlay

The M4 Colab overlay is installed into a private work venv with system site packages and hashed wheels. It excludes Torch/NVIDIA packages. The notebook checks the native Torch version and file path remain unchanged.



In [ ]:
import subprocess

if not VENV.exists():
    subprocess.run(
        [sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(VENV)],
        check=True,
        timeout=max(1, min(120, DIAGNOSTIC_DEADLINE - time.time() - 300)),
    )
VENV_PYTHON = VENV / "bin/python"
subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "--version"],
    check=True,
    timeout=max(1, min(120, DIAGNOSTIC_DEADLINE - time.time() - 300)),
)
requirements = INPUTS / "environments/colab/requirements.txt"
subprocess.run(
    [
        str(VENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-cache-dir",
        "--no-deps",
        "--require-hashes",
        "--only-binary=:all:",
        "-r",
        str(requirements),
    ],
    check=True,
    timeout=max(1, min(120, DIAGNOSTIC_DEADLINE - time.time() - 300)),
)
probe = subprocess.run(
    [
        str(VENV_PYTHON),
        "-c",
        "import torch,sys; print(sys.version); print(torch.__version__); "
        "print(torch.version.cuda); print(torch.__file__)",
    ],
    check=True,
    capture_output=True,
    text=True,
    timeout=max(1, min(60, DIAGNOSTIC_DEADLINE - time.time() - 300)),
).stdout.splitlines()
print("Training environment:", probe)
if probe[1] != "2.11.0+cu130" or probe[2] != "13.0" or probe[3] != native_torch_path:
    raise RuntimeError("Pinned overlay changed or cannot see the native CUDA Torch build")

In [ ]:
import os
import shutil

from google.colab import drive

drive.mount("/content/drive")
PERSISTENT = Path(approved["persistent_destination"])
if not str(PERSISTENT).startswith("/content/drive/MyDrive/AtlasRank/m5/"):
    raise PermissionError("Only the approved private AtlasRank M5 Drive subtree is allowed")
if not approved.get("operator_owner_token"):
    raise PermissionError("Approved manifest lacks the exclusive runtime token")
if PERSISTENT.exists() and any(PERSISTENT.iterdir()):
    raise PermissionError("Diagnostic destination must be new and empty; no implicit retry")
inputs_drive = PERSISTENT / "inputs"
inputs_drive.mkdir(parents=True, exist_ok=True)
approved_copy = PERSISTENT / "approved-m5-pilot.json"
drive_archive = inputs_drive / "m5-training-input-v8.zip"
for source, target in ((APPROVED_MANIFEST, approved_copy), (ARCHIVE_UPLOAD, drive_archive)):
    expected = hashlib.sha256(source.read_bytes()).hexdigest()
    if target.exists():
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError(f"Preserve existing Drive file; checksum differs: {target.name}")
    else:
        with target.open("xb") as destination, source.open("rb") as origin:
            shutil.copyfileobj(origin, destination)
os.environ["ATLAS_COLAB_EXCLUSIVE_RUNTIME"] = approved["operator_owner_token"]
print("Private destination:", PERSISTENT)
print("Input export persisted and hash-checked; package runner creates the owner lock.")

## Training
Run the existing package once. It validates the ten-update GPU sanity check and saves its checkpoint before continuing the same optimizer/scheduler/RNG/sampler state to 574 updates. Failures stop execution and preserve diagnostics. The completed destination must never be implicitly replayed.


In [ ]:
import hashlib
import json
import time

checkpoints = PERSISTENT / "checkpoints"
if any(checkpoints.rglob("step-*.pt")):
    raise PermissionError("No implicit replay of a completed or interrupted destination")
command = [
    str(VENV_PYTHON),
    str(INPUTS / "source/scripts/run_m5_training.py"),
    "--archive",
    str(ARCHIVE_UPLOAD),
    "--manifest",
    str(APPROVED_MANIFEST),
    "--model",
    str(INPUTS / "model"),
    "--input-root",
    str(INPUTS),
    "--checkpoints",
    str(checkpoints),
    "--persistent",
    str(PERSISTENT),
    "--allocation-started-utc",
    str(ALLOCATION_STARTED_UTC),
]
remaining = DIAGNOSTIC_DEADLINE - time.time()
if remaining < 600:
    raise TimeoutError("Need 300 active plus 300 finalization seconds; no diagnostic launched")
run_env = os.environ.copy()
run_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
run_env["PYTHONPATH"] = str(INPUTS / "source/src")
runner_log = WORK / "diagnostic-runner.log"
try:
    with runner_log.open("xb") as stream:
        subprocess.run(
            command,
            check=True,
            env=run_env,
            cwd=WORK,
            stdout=stream,
            stderr=subprocess.STDOUT,
            timeout=min(2760, remaining - 300),
        )
finally:
    # Preserve stderr even when the output iframe fails or the child times out.
    target = PERSISTENT / "diagnostic-runner.log"
    with target.open("xb") as out, runner_log.open("rb") as source:
        shutil.copyfileobj(source, out)
    digest = hashlib.sha256(target.read_bytes()).hexdigest()
    (PERSISTENT / "diagnostic-runner.receipt.json").write_text(
        json.dumps(
            {
                "sha256": digest,
                "bytes": target.stat().st_size,
                "allocation_elapsed_seconds": time.time() - ALLOCATION_STARTED_UTC,
            }
        )
    )
    print("Runner log persisted:", digest)

In [ ]:
# Completed training only; reports and final snapshot are already in private Drive.
records = list(PERSISTENT.glob("*.run.json"))
if len(records) != 1:
    raise RuntimeError("No unique completed training record; preserve failure and release manually")
result = json.loads(records[0].read_text())
if result["status"] != "completed" or result["global_step"] != 574:
    raise RuntimeError("Training did not complete exactly 574 updates")
print("Training completed; private Drive artifacts retained. Releasing ephemeral L4 VM.")
print("Elapsed allocation seconds:", time.time() - ALLOCATION_STARTED_UTC)
runtime.unassign()